In [0]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

import os
import warnings
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")


In [0]:
import sys
import pyspark.sql.functions as F
from src.config import data_paths, filter_map
from src.data_ingestion import DataLoader,DataMerger,Preprocessor
from src.logger import logger
from src.exception import M5Exception
from src.utils import spark

In [0]:
prod_cus_dim_path = data_paths['bronze']['processed_data']['product_customer_dim']
joined_data_path = data_paths['bronze']['processed_data']['joined_data']

In [0]:
data = pd.read_parquet(joined_data_path)
data = data.sort_values(by=['id','yearweek'])
data.head()

In [0]:
prod_cus_dim = pd.read_parquet(prod_cus_dim_path)
prod_cus_dim.head()

In [0]:
joined_data = data.merge(prod_cus_dim,on=['id'],how='inner')
joined_data.info()


In [0]:
#how many products[items], customers[stores], Categories?
joined_data['id'].nunique(), joined_data['item_id'].nunique(),joined_data['store_id'].nunique(),joined_data['cat_id'].nunique()

In [0]:
joined_data.groupby(['state_id','cat_id'],as_index=False).agg(unique_ids=('id','nunique'),
                                          avg_sales = ('sales','mean'),
                                          total_sales = ('sales','sum'),
                                          sales_std = ('sales','std'),
                                          total_weeks = ('yearweek','nunique'))\
                                    .sort_values(by=['unique_ids','avg_sales'],ascending=False)\
                                    .assign(total_sales_m = lambda df:df['total_sales']/1e6,
                                            sales_pct = lambda df: (df['total_sales']/df['total_sales'].sum()) * 100,
                                            id_pct = lambda df: (df['unique_ids']/df['unique_ids'].sum())*100)

- 67% of sales, 47% of series are from Foods
- 22% of sales, 34% of series are from HouseHold
- 11% of sales, 18% of series are from Hobbies

In [0]:

ID, CTRY, CAT, WEEK, Y, PRICE = "id", "state_id", "cat_id", "yearweek", "sales", "sell_price"
# SEAS_STRENGTH_MIN = 0.4   # STL seasonal strength threshold
# SEAS_CORR_MIN = 0.5       # year-over-year weekly profile correlation threshold


# def _nan_corr(X, Z, min_pairs):
#     """column-wise correlation ignoring NaN pairs"""
#     m = ~np.isnan(X) & ~np.isnan(Z)
#     X, Z = np.where(m, X, np.nan), np.where(m, Z, np.nan)
#     dx, dz = X - np.nanmean(X, 0), Z - np.nanmean(Z, 0)
#     den = np.sqrt(np.nansum(dx**2, 0) * np.nansum(dz**2, 0))
#     out = np.where(den > 0, np.nansum(dx * dz, 0) / den, np.nan)
#     return np.where(m.sum(0) >= min_pairs, out, np.nan)

# def _stl_seasonal_strength(x, period=52):
#     from statsmodels.tsa.seasonal import STL
#     x = pd.Series(x).interpolate().dropna()
#     if len(x) < 2 * period + 1:
#         return np.nan
#     try:
#         r = STL(np.log1p(x.to_numpy()), period=period, robust=True).fit()
#     except Exception:
#         return np.nan
#     return max(0.0, 1 - np.var(r.resid) / np.var(r.seasonal + r.resid))


# def _yearly_profile_corr(W):
#     """For each id: correlation of each year's weekly profile (weeks 1-52) with the
#     previous year's. Returns mean corr, number of year pairs, and share of pairs above threshold."""
#     L = np.log1p(W)
#     iso = L.index.isocalendar()
#     yr, wk = iso.year.to_numpy(), iso.week.to_numpy()
#     blocks = {}
#     for y in sorted(set(yr)):
#         m = (yr == y) & (wk <= 52)
#         blocks[y] = L.loc[m].set_axis(wk[m]).reindex(range(1, 53)).to_numpy()
#     ys = sorted(blocks)
#     cors = np.array([_nan_corr(blocks[b], blocks[a], min_pairs=40) for a, b in zip(ys[:-1], ys[1:])])
#     n_pairs = (~np.isnan(cors)).sum(0)
#     return (np.nanmean(cors, 0), n_pairs, np.where(n_pairs > 0, np.nansum(cors >= SEAS_CORR_MIN, 0) / np.maximum(n_pairs, 1), np.nan))



# def build_descriptors(df, stl=True):
#     df = df.copy()
#     df["date"] = pd.to_datetime(df[WEEK].astype(str) + "1", format="%G%V%u")  # ISO weeks
 
#     W = df.pivot(index="date", columns=ID, values=Y).sort_index()
#     W = W.reindex(pd.date_range(W.index.min(), W.index.max(), freq="7D"))
#     A = W.to_numpy(float)
 
#     obs = ~np.isnan(A)
#     pos = obs & (A > 0)
#     n, n_pos = obs.sum(0), pos.sum(0)
#     A_pos = np.where(pos, A, np.nan)
 
#     d = pd.DataFrame(index=W.columns)
#     d.index.name = ID
 
#     # size and history
#     d["history_len"] = n
#     d["mean_sales"] = np.nanmean(A, 0)
#     d["vol_52w"] = np.nansum(A[-52:], 0)
 
#     # intermittency
#     d["zero_share"] = 1 - n_pos / n
#     d["adi"] = np.where(n_pos > 0, n / np.maximum(n_pos, 1), np.inf)
#     d["cv2"] = (np.nanstd(A_pos, 0) / np.nanmean(A_pos, 0)) ** 2
 
#     # growth and lifecycle
#     l52, p52 = np.nanmean(A[-52:], 0), np.nanmean(A[-104:-52], 0)
#     d["yoy_ratio"] = np.where(p52 > 0, l52 / p52, np.nan)
#     d["tail_zero_weeks"] = np.cumprod(A[::-1] == 0, axis=0).sum(0)
 
#     # seasonality
#     L = np.log1p(A)
#     d["acf52"] = _nan_corr(L[52:], L[:-52], min_pairs=26)
#     d["yr_profile_corr"], d["n_year_pairs"], d["share_years_seasonal"] = _yearly_profile_corr(W)
#     if stl:
#         d["seasonal_strength"] = Parallel(n_jobs=-1, batch_size=256)(
#             delayed(_stl_seasonal_strength)(A[:, j]) for j in range(A.shape[1]))
#     else:
#         d["seasonal_strength"] = np.nan
 
#     # price
#     p = df[PRICE].where(df[PRICE] > 0)
#     g = df[ID]
#     d["price_cv"] = p.groupby(g).std() / p.groupby(g).mean()
#     d["promo_share"] = (p < 0.9 * p.groupby(g).transform("median")).groupby(g).mean()
 
#     # labels
#     d = d.join(df.groupby(ID)[[CTRY, CAT]].first())
#     d["is_new"] = d.history_len < 52
#     d["is_dead"] = d.tail_zero_weeks >= 13
#     d["demand_class"] = np.select(
#         [(d.adi < 1.32) & (d.cv2 < 0.49), (d.adi < 1.32) & (d.cv2 >= 0.49),
#          (d.adi >= 1.32) & (d.cv2 < 0.49)],
#         ["smooth", "erratic", "intermittent", ][:3], "lumpy")
 
#     # seasonal class: needs >= 2 years of history; seasonal only if BOTH the STL strength
#     # and the year-over-year profile correlation agree (and it is not an intermittent/dead series)
#     can_test = (d.history_len >= 104) & d.seasonal_strength.notna() & (d.n_year_pairs >= 1)
#     is_seas = (d.seasonal_strength >= SEAS_STRENGTH_MIN) & (d.yr_profile_corr >= SEAS_CORR_MIN)
#     d["seasonal_class"] = np.select([~can_test, is_seas], ["unknown", "seasonal"], "non_seasonal")
 
#     d["segment"] = d.demand_class + "_" + d.seasonal_class
 
#     # ABC within country x category on last-52-week volume
#     d = d.sort_values("vol_52w", ascending=False)
#     share = d.groupby([CTRY, CAT])["vol_52w"].transform(lambda s: s.cumsum() / s.sum())
#     d["abc"] = np.select([share <= 0.8, share <= 0.95], ["A", "B"], "C")
#     return d.sort_index()


SEAS_CORR_MIN = 0.5       # leave-one-year-out profile correlation (null sd ~0.2-0.3)
SEAS_AMP_MIN = 0.10       # seasonal swing in log units (~ +/-10%)


def _nan_corr(X, Z, min_pairs):
    """column-wise correlation ignoring NaN pairs"""
    m = ~np.isnan(X) & ~np.isnan(Z)
    X, Z = np.where(m, X, np.nan), np.where(m, Z, np.nan)
    dx, dz = X - np.nanmean(X, 0), Z - np.nanmean(Z, 0)
    den = np.sqrt(np.nansum(dx**2, 0) * np.nansum(dz**2, 0))
    out = np.where(den > 0, np.nansum(dx * dz, 0) / den, np.nan)
    return np.where(m.sum(0) >= min_pairs, out, np.nan)


def seasonal_features(W, min_years=3, min_buckets=9):
    """Cheap, trend-robust seasonality test (no model fitting).

    1. log1p(sales) minus centred 52-week moving average  -> removes level and trend
    2. average into 13 four-week buckets per ISO year      -> removes weekly noise
    3. leave-one-year-out: correlate each year's bucket profile with the average of the
       other years, then average over years                -> "does last year's shape repeat?"
    4. amplitude = std of the average profile (log units)  -> "is the swing big enough to matter?"
    """
    L = np.log1p(W)
    ma = L.rolling(52).mean()
    x = L - (ma.shift(-25) + ma.shift(-26)) / 2          # centred 2x52 moving average

    iso = x.index.isocalendar()
    yr = iso.year.to_numpy(dtype=int)
    bucket = np.minimum((iso.week.to_numpy(dtype=int) - 1) // 4, 12)
    P = x.groupby([yr, bucket]).mean().where(x.notna().groupby([yr, bucket]).sum() >= 3)
    ys = sorted(set(yr))
    arr = np.stack([P.loc[y].reindex(range(13)).to_numpy() for y in ys])   # years x 13 x N
    arr = arr - np.nanmean(arr, axis=1, keepdims=True)
    valid = (~np.isnan(arr)).sum(1) >= min_buckets                          # years x N
    arr = np.where(valid[:, None, :], arr, np.nan)

    cors = []
    for i in range(len(ys)):
        others = np.nanmean(np.delete(arr, i, 0), axis=0)
        c = _nan_corr(arr[i], others, min_pairs=min_buckets)
        cors.append(np.where(np.delete(valid, i, 0).sum(0) >= 2, c, np.nan))
    out = pd.DataFrame(index=W.columns)
    out["n_seas_years"] = valid.sum(0)
    out["seas_corr"] = np.nanmean(np.array(cors), axis=0)
    out["seas_amp"] = np.nanstd(np.nanmean(arr, axis=0), axis=0)
    return out


def build_descriptors(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df[WEEK].astype(str) + "1", format="%G%V%u")  # ISO weeks

    W = df.pivot(index="date", columns=ID, values=Y).sort_index()
    W = W.reindex(pd.date_range(W.index.min(), W.index.max(), freq="7D"))
    A = W.to_numpy(float)

    obs = ~np.isnan(A)
    pos = obs & (A > 0)
    n, n_pos = obs.sum(0), pos.sum(0)
    A_pos = np.where(pos, A, np.nan)

    d = pd.DataFrame(index=W.columns)
    d.index.name = ID

    # size and history
    d["age_weeks"] = n
    d["mean_sales"] = np.nanmean(A, 0)
    d["vol_52w"] = np.nansum(A[-52:], 0)

    # intermittency
    d["zero_share"] = 1 - n_pos / n
    d["adi"] = np.where(n_pos > 0, n / np.maximum(n_pos, 1), np.inf)
    d["cv2"] = (np.nanstd(A_pos, 0) / np.nanmean(A_pos, 0)) ** 2

    # growth and lifecycle
    l52, p52 = np.nanmean(A[-52:], 0), np.nanmean(A[-104:-52], 0)
    d["yoy_ratio"] = np.where(p52 > 0, l52 / p52, np.nan)
    d["tail_zero_weeks"] = np.cumprod(A[::-1] == 0, axis=0).sum(0)

    # seasonality
    L = np.log1p(A)
    d["acf52"] = _nan_corr(L[52:], L[:-52], min_pairs=26)
    d = d.join(seasonal_features(W))

    # price
    p = df[PRICE].where(df[PRICE] > 0)
    g = df[ID]
    d["price_cv"] = p.groupby(g).std() / p.groupby(g).mean()
    d["promo_share"] = (p < 0.9 * p.groupby(g).transform("median")).groupby(g).mean()

    # labels
    d = d.join(df.groupby(ID)[[CTRY, CAT]].first())
    d["is_new"] = d.age_weeks < 52
    d["is_dead"] = d.tail_zero_weeks >= 13
    d["demand_class"] = np.select(
        [(d.adi < 1.32) & (d.cv2 < 0.49), (d.adi < 1.32) & (d.cv2 >= 0.49),
         (d.adi >= 1.32) & (d.cv2 < 0.49)],
        ["smooth", "erratic", "intermittent"], "lumpy")

    # seasonal class: needs >= 3 usable years; seasonal = shape repeats AND swing is material
    d["seasonal_class"] = np.select(
        [d.n_seas_years < 3, (d.seas_corr >= SEAS_CORR_MIN) & (d.seas_amp >= SEAS_AMP_MIN)],
        ["unknown", "seasonal"], "non_seasonal")

    d["segment"] = d.demand_class + "_" + d.seasonal_class

    # ABC within country x category on last-52-week volume
    d = d.sort_values("vol_52w", ascending=False)
    share = d.groupby([CTRY, CAT])["vol_52w"].transform(lambda s: s.cumsum() / s.sum())
    d["abc"] = np.select([share <= 0.8, share <= 0.95], ["A", "B"], "C")
    return d.sort_index()

 

In [0]:
data_cube = build_descriptors(joined_data)

In [0]:
data_cube

In [0]:
data_cube.groupby(['state_id','cat_id']).agg(
                                            delisted_id_count = ('is_dead','sum'),
                                            npd_id_count = ('is_new','sum')
                                             )

In [0]:
data_cube.groupby(["state_id", "cat_id", "demand_class"]).size().unstack()

In [0]:
data_cube.groupby(["state_id", "cat_id", "seasonal_class"]).size().unstack()

In [0]:
data_cube.groupby(["state_id", "cat_id", "segment"]).size().unstack()

In [0]:
data_cube.groupby("demand_class",as_index=False)["vol_52w"].sum().sort_values(by=['vol_52w'],ascending=False)